# Module 1 — Data Pipeline

This module builds an end-to-end data pipeline using
books.toscrape.com. The workflow covers web scraping,
data cleaning, currency conversion, SQLite database
storage, SQL analysis, and pandas validation.

In [69]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
import sqlite3

BASE_URL = "https://books.toscrape.com/"

# Get category links
response = requests.get(BASE_URL)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

category_links = {}

for link in soup.select(".side_categories ul li ul li a"):
    category_name = link.get_text(strip=True)
    category_url = urljoin(BASE_URL, link.get("href"))
    category_links[category_name] = category_url

# Select three categories
selected_categories = [
    "Travel",
    "Mystery",
    "Historical Fiction"
]

scraped_books = []

for category in selected_categories:

    page_url = category_links[category]

    while page_url:

        response = requests.get(page_url)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        books = soup.select("article.product_pod")

        for book in books:

            title = book.h3.a.get("title")
            price = book.select_one(".price_color").get_text(strip=True)
            star_rating = book.select_one(".star-rating")["class"][1]
            availability = book.select_one(".availability").get_text(" ", strip=True)

            scraped_books.append({
                "title": title,
                "price": price,
                "star_rating": star_rating,
                "availability": availability,
                "category": category
            })

        # Find the next page
        next_link = soup.select_one("li.next a")

        if next_link:
            page_url = urljoin(page_url, next_link.get("href"))
        else:
            page_url = None


# Convert to DataFrame
books_df = pd.DataFrame(scraped_books)

print("Total books scraped:", len(books_df))
print("Number of categories:", books_df["category"].nunique())

# Validate Task 1 requirements
assert len(books_df) >= 60
assert books_df["category"].nunique() >= 3


books_df.head()

Total books scraped: 69
Number of categories: 3


,title,price,star_rating,availability,category
0,It's Only the Himalayas,Â£45.17,Two,In stock,Travel
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,Â£49.43,Four,In stock,Travel
2,See America: A Celebration of Our National Par...,Â£48.87,Three,In stock,Travel
3,Vagabonding: An Uncommon Guide to the Art of L...,Â£36.94,Two,In stock,Travel
4,Under the Tuscan Sun,Â£37.33,Three,In stock,Travel


Task 2


In [21]:
# Create a copy so that the original scraped data remains unchanged
cleaned_df = books_df.copy()

# 1. Clean price and convert it to a numeric value
cleaned_df["price_gbp"] = (
    cleaned_df["price"]
    .str.replace(r"[^\d.]", "", regex=True)
    .pipe(pd.to_numeric, errors="coerce")
)
# If any price cannot be converted, replace it with the median price
invalid_prices = cleaned_df["price_gbp"].isna().sum()

if invalid_prices > 0:
    median_price = cleaned_df["price_gbp"].median()
    cleaned_df["price_gbp"] = cleaned_df["price_gbp"].fillna(median_price)

# 2. Convert star ratings from words to numbers
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

cleaned_df["rating"] = cleaned_df["star_rating"].map(rating_map)

# Handle any unexpected rating values using the median
invalid_ratings = cleaned_df["rating"].isna().sum()

if invalid_ratings > 0:
    median_rating = cleaned_df["rating"].median()
    cleaned_df["rating"] = cleaned_df["rating"].fillna(median_rating)

cleaned_df["rating"] = cleaned_df["rating"].astype(int)

# 3. Convert availability into a boolean value
cleaned_df["in_stock"] = (
    cleaned_df["availability"]
    .str.strip()
    .str.lower()
    .str.startswith("in stock")
)

# Display the cleaned data
cleaned_df[
    [
        "title",
        "price_gbp",
        "availability",
        "rating",
        "in_stock",
        "category"
    ]
].head(50)

,title,price_gbp,availability,rating,in_stock,category
0,It's Only the Himalayas,45.17,In stock,2,True,Travel
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,49.43,In stock,4,True,Travel
2,See America: A Celebration of Our National Par...,48.87,In stock,3,True,Travel
3,Vagabonding: An Uncommon Guide to the Art of L...,36.94,In stock,2,True,Travel
4,Under the Tuscan Sun,37.33,In stock,3,True,Travel
5,A Summer In Europe,44.34,In stock,2,True,Travel
6,The Great Railway Bazaar,30.54,In stock,1,True,Travel
7,A Year in Provence (Provence #1),56.88,In stock,4,True,Travel
8,The Road to Little Dribbling: Adventures of an...,23.21,In stock,1,True,Travel
9,Neither Here nor There: Travels in Europe,38.95,In stock,3,True,Travel


Task 3


In [22]:
# Task 3 - Convert GBP prices to INR

# Fixed conversion rate provided by the project
GBP_TO_INR = 105.50

print(f"Using fixed conversion rate: 1 GBP = {GBP_TO_INR} INR")

# Calculate the INR price from the cleaned GBP price
cleaned_df["price_inr"] = cleaned_df["price_gbp"] * GBP_TO_INR

# Round the result to two decimal places for currency presentation
cleaned_df["price_inr"] = cleaned_df["price_inr"].round(2)

# Display a few converted prices
print("\nSample of converted prices:")
display(
    cleaned_df[
        ["title", "price_gbp", "price_inr"]
    ].head(10)
)

# Check for any missing values after conversion
missing_inr = cleaned_df["price_inr"].isna().sum()
print(f"\nMissing INR values: {missing_inr}")

# Verify the conversion for the first row
first_gbp = cleaned_df.loc[0, "price_gbp"]
expected_inr = round(first_gbp * GBP_TO_INR, 2)
actual_inr = cleaned_df.loc[0, "price_inr"]

print("\nConversion check:")
print(f"GBP price: {first_gbp}")
print(f"Expected INR price: {expected_inr}")
print(f"Stored INR price: {actual_inr}")

Using fixed conversion rate: 1 GBP = 105.5 INR

Sample of converted prices:


,title,price_gbp,price_inr
0,It's Only the Himalayas,45.17,4765.44
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,49.43,5214.86
2,See America: A Celebration of Our National Par...,48.87,5155.78
3,Vagabonding: An Uncommon Guide to the Art of L...,36.94,3897.17
4,Under the Tuscan Sun,37.33,3938.31
5,A Summer In Europe,44.34,4677.87
6,The Great Railway Bazaar,30.54,3221.97
7,A Year in Provence (Provence #1),56.88,6000.84
8,The Road to Little Dribbling: Adventures of an...,23.21,2448.66
9,Neither Here nor There: Travels in Europe,38.95,4109.23



Missing INR values: 0

Conversion check:
GBP price: 45.17
Expected INR price: 4765.44
Stored INR price: 4765.44


Task 4


In [25]:
# Create a connection to the SQLite database
db_name = "books.db"

connection = sqlite3.connect(db_name)

print(f"Connected to {db_name}")

Connected to books.db


In [26]:
# Make sure SQLite enforces foreign-key relationships
connection.execute("PRAGMA foreign_keys = ON")

print("Foreign-key support enabled.")

Foreign-key support enabled.


In [27]:
create_categories_table = """
CREATE TABLE IF NOT EXISTS categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT NOT NULL UNIQUE
);
"""

connection.execute(create_categories_table)
connection.commit()

print("Categories table is ready.")

Categories table is ready.


In [30]:
create_books_table = """
CREATE TABLE IF NOT EXISTS books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    price_gbp REAL,
    price_inr REAL,
    rating INTEGER,
    in_stock INTEGER NOT NULL,
    category_id INTEGER NOT NULL,
    FOREIGN KEY (category_id)
        REFERENCES categories(category_id)
);
"""

connection.execute(create_books_table)
connection.commit()

print("Books table is ready.")

Books table is ready.


In [32]:
category_names = cleaned_df["category"].dropna().unique()

print("Categories found:", len(category_names))
print(category_names)

Categories found: 3
['Travel' 'Mystery' 'Historical Fiction']


In [33]:
insert_category = """
INSERT OR IGNORE INTO categories (category_name)
VALUES (?);
"""

for category in category_names:
    connection.execute(insert_category, (category,))

connection.commit()

print("Categories inserted successfully.")

Categories inserted successfully.


In [35]:
category_lookup = dict(
    zip(
        categories_df["category_name"],
        categories_df["category_id"]
    )
)

category_lookup

{'Travel': 1, 'Mystery': 2, 'Historical Fiction': 3}

In [36]:
cleaned_df["category_id"] = cleaned_df["category"].map(category_lookup)

In [39]:
insert_book = """
INSERT INTO books (
    title,
    price_gbp,
    price_inr,
    rating,
    in_stock,
    category_id
)
VALUES (?, ?, ?, ?, ?, ?);
"""

for _, row in cleaned_df.iterrows():
    connection.execute(
        insert_book,
        (
            row["title"],
            row["price_gbp"],
            row["price_inr"],
            row["rating"],
            int(row["in_stock"]),
            row["category_id"]
        )
    )

connection.commit()

print(f"{len(cleaned_df)} books inserted into the database.")

69 books inserted into the database.


In [40]:
books_from_db = pd.read_sql(
    """
    SELECT *
    FROM books
    LIMIT 10;
    """,
    connection
)

books_from_db

,book_id,title,price_gbp,price_inr,rating,in_stock,category_id
0,1,It's Only the Himalayas,45.17,4765.44,2,1,1
1,2,Full Moon over Noahâs Ark: An Odyssey to Mou...,49.43,5214.86,4,1,1
2,3,See America: A Celebration of Our National Par...,48.87,5155.78,3,1,1
3,4,Vagabonding: An Uncommon Guide to the Art of L...,36.94,3897.17,2,1,1
4,5,Under the Tuscan Sun,37.33,3938.31,3,1,1
5,6,A Summer In Europe,44.34,4677.87,2,1,1
6,7,The Great Railway Bazaar,30.54,3221.97,1,1,1
7,8,A Year in Provence (Provence #1),56.88,6000.84,4,1,1
8,9,The Road to Little Dribbling: Adventures of an...,23.21,2448.66,1,1,1
9,10,Neither Here nor There: Travels in Europe,38.95,4109.23,3,1,1


In [47]:
connection.close()

print("Database connection closed.")

Database connection closed.


Task 5

In [48]:
import sqlite3
import pandas as pd

connection = sqlite3.connect("books.db")

print("Connected to the books database.")

Connected to the books database.


In [49]:
# Query 1 - SELECT + WHERE
query_1 = """
SELECT title, price_gbp, rating
FROM books
WHERE rating >= 4;
"""

result_1 = pd.read_sql(query_1, connection)
print("Query 1: Books with a rating of 4 or higher")
display(result_1)

Query 1: Books with a rating of 4 or higher


,title,price_gbp,rating
0,Full Moon over Noahâs Ark: An Odyssey to Mou...,49.43,4
1,A Year in Provence (Provence #1),56.88,4
2,"1,000 Places to See Before You Die",26.08,5
3,Sharp Objects,47.82,4
4,The Past Never Ends,56.50,4
5,The Murder of Roger Ackroyd (Hercule Poirot #4),44.10,4
6,A Time of Torment (Charlie Parker #14),48.35,5
7,Murder at the 42nd Street Library (Raymond Amb...,54.36,4
8,What Happened on Beale Street (Secrets of the ...,25.37,5
9,The Bachelor Girl's Guide to Murder (Herringfo...,52.30,5


In [50]:
# Query 2 - ORDER BY
query_2 = """
SELECT title, price_gbp, rating
FROM books
ORDER BY price_gbp DESC;
"""

result_2 = pd.read_sql(query_2, connection)
print("Query 2: Books ordered by price")
display(result_2)

Query 2: Books ordered by price


,title,price_gbp,rating
0,Boar Island (Anna Pigeon #19),59.48,3
1,The No. 1 Ladies' Detective Agency (No. 1 Ladi...,57.70,4
2,A Year in Provence (Provence #1),56.88,4
3,The Past Never Ends,56.50,4
4,The Last Painting of Sara de Vos,55.55,2
...,...,...,...
64,That Darkness (Gardiner and Renner #1),13.92,1
65,Playing with Fire,13.71,3
66,The Girl You Lost,12.29,5
67,Hide Away (Eve Duncan #20),11.84,1


In [51]:
# Query 3 - LIMIT
query_3 = """
SELECT title, price_gbp, rating
FROM books
ORDER BY price_gbp DESC
LIMIT 10;
"""

result_3 = pd.read_sql(query_3, connection)
print("Query 3: Top 10 most expensive books")
display(result_3)

Query 3: Top 10 most expensive books


,title,price_gbp,rating
0,Boar Island (Anna Pigeon #19),59.48,3
1,The No. 1 Ladies' Detective Agency (No. 1 Ladi...,57.70,4
2,A Year in Provence (Provence #1),56.88,4
3,The Past Never Ends,56.50,4
4,The Last Painting of Sara de Vos,55.55,2
5,A Flight of Arrows (The Pathfinders #2),55.53,5
6,Murder at the 42nd Street Library (Raymond Amb...,54.36,4
7,The Last Mile (Amos Decker #2),54.21,2
8,1st to Die (Women's Murder Club #1),53.98,1
9,Tipping the Velvet,53.74,1


In [52]:
# Query 4 - DISTINCT
query_4 = """
SELECT DISTINCT rating
FROM books
ORDER BY rating;
"""

result_4 = pd.read_sql(query_4, connection)
print("Query 4: Different ratings available")
display(result_4)

Query 4: Different ratings available


,rating
0,1
1,2
2,3
3,4
4,5


In [53]:
# Query 5 - BETWEEN
query_5 = """
SELECT title, price_gbp, price_inr
FROM books
WHERE price_gbp BETWEEN 20 AND 40
ORDER BY price_gbp;
"""

result_5 = pd.read_sql(query_5, connection)
print("Query 5: Books priced between £20 and £40")
display(result_5)

Query 5: Books priced between £20 and £40


,title,price_gbp,price_inr
0,Blood Defense (Samantha Brinkman #1),20.30,2141.65
1,"Love, Lies and Spies",20.55,2168.02
2,Between Shades of Gray,20.79,2193.34
3,Delivering the Truth (Quaker Midwife Mystery #1),20.89,2203.90
4,Voyager (Outlander #3),21.07,2222.89
5,The Silkworm (Cormoran Strike #2),23.05,2431.78
6,The Road to Little Dribbling: Adventures of an...,23.21,2448.66
7,Career of Evil (Cormoran Strike #3),24.72,2607.96
8,The Mysterious Affair at Styles (Hercule Poiro...,24.80,2616.40
9,What Happened on Beale Street (Secrets of the ...,25.37,2676.54


In [54]:
# Query 6 - JOIN
query_6 = """
SELECT
    b.title,
    c.category_name,
    b.price_gbp,
    b.rating,
    b.in_stock
FROM books AS b
JOIN categories AS c
    ON b.category_id = c.category_id
ORDER BY b.rating DESC, b.price_gbp DESC
LIMIT 10;
"""

result_6 = pd.read_sql(query_6, connection)
print("Query 6: Top-rated books with their categories")
display(result_6)

Query 6: Top-rated books with their categories


,title,category_name,price_gbp,rating,in_stock
0,A Flight of Arrows (The Pathfinders #2),Historical Fiction,55.53,5,1
1,The Bachelor Girl's Guide to Murder (Herringfo...,Mystery,52.30,5,1
2,A Time of Torment (Charlie Parker #14),Mystery,48.35,5,1
3,While You Were Mine,Historical Fiction,41.32,5,1
4,The Red Tent,Historical Fiction,35.66,5,1
5,Mrs. Houdini,Historical Fiction,30.25,5,1
6,The Passion of Dolssa,Historical Fiction,28.32,5,1
7,"1,000 Places to See Before You Die",Travel,26.08,5,1
8,What Happened on Beale Street (Secrets of the ...,Mystery,25.37,5,1
9,The Silkworm (Cormoran Strike #2),Mystery,23.05,5,1


In [57]:
queries_and_results = [
    ("Query 1 - SELECT and WHERE", query_1, result_1),
    ("Query 2 - ORDER BY", query_2, result_2),
    ("Query 3 - LIMIT", query_3, result_3),
    ("Query 4 - DISTINCT", query_4, result_4),
    ("Query 5 - BETWEEN", query_5, result_5),
    ("Query 6 - JOIN", query_6, result_6)
]

with open("query_outputs.txt", "w", encoding="utf-8") as file:
    for name, query, result in queries_and_results:
        file.write("=" * 60 + "\n")
        file.write(name + "\n")
        file.write("=" * 60 + "\n\n")

        file.write("SQL:\n")
        file.write(query.strip() + "\n\n")

        file.write("Output:\n")
        file.write(result.to_string(index=False))
        file.write("\n\n")

print("Queries and outputs saved to query_outputs.txt")

Queries and outputs saved to query_outputs.txt


In [56]:
sql_text = ""

for name, query, result in queries_and_results:
    sql_text += f"-- {name}\n"
    sql_text += query.strip() + "\n\n"

with open("queries.sql", "w", encoding="utf-8") as file:
    file.write(sql_text)

print("SQL queries saved to queries.sql")
connection.close()

print("Database connection closed.")

SQL queries saved to queries.sql
Database connection closed.


Task 6

In [68]:
# Connect to the SQLite database
connection = sqlite3.connect("books.db")

print("Connected to the database.")

sql_result_1 = pd.read_sql(query_1, connection)

sql_result_5 = pd.read_sql(query_5, connection)

print("\nQuery 1 result:")
display(sql_result_1.head(10))

print("\nQuery 5 result:")
display(sql_result_5.head(10))

# 2.Load the original tables into pandas


books_data = pd.read_sql(
    "SELECT * FROM books;",
    connection
)

categories_data = pd.read_sql(
    "SELECT * FROM categories;",
    connection
)

print("\nBooks table:")
display(books_data.head())

print("\nCategories table:")
display(categories_data.head())


# 3.Reproduce the SQL JOIN using pandas merge


merged_data = pd.merge(
    books_data,
    categories_data,
    on="category_id",
    how="inner"
)

# Keep the same columns used by the SQL JOIN
merge_result = merged_data[
    [
        "title",
        "category_name",
        "price_gbp",
        "rating",
        "in_stock"
    ]
].copy()

# Reproduce the SQL ORDER BY and LIMIT
merge_result = (
    merge_result
    .sort_values(
        by=["rating", "price_gbp"],
        ascending=[False, False]
    )
    .head(10)
    .reset_index(drop=True)
)


# 4. Prepare the SQL JOIN result for comparison


sql_join_result = result_6[
    [
        "title",
        "category_name",
        "price_gbp",
        "rating",
        "in_stock"
    ]
].copy()

sql_join_result = sql_join_result.reset_index(drop=True)


# 5. Display both results


print("\nJOIN result using SQL:")
display(sql_join_result)

print("\nJOIN result using pandas merge:")
display(merge_result)


# 6. Compare the two results


results_match = sql_join_result.equals(merge_result)

print("\nDo both approaches produce the same result?")
print(results_match)


# 7. Show the results side by side


side_by_side = pd.concat(
    [
        sql_join_result.add_prefix("SQL_"),
        merge_result.add_prefix("Pandas_")
    ],
    axis=1
)

print("\nSQL JOIN vs pandas merge:")
display(side_by_side)


# Close the database connection
connection.close()

print("\nDatabase connection closed.")

Connected to the database.

Query 1 result:


,title,price_gbp,rating
0,Full Moon over Noahâs Ark: An Odyssey to Mou...,49.43,4
1,A Year in Provence (Provence #1),56.88,4
2,"1,000 Places to See Before You Die",26.08,5
3,Sharp Objects,47.82,4
4,The Past Never Ends,56.50,4
5,The Murder of Roger Ackroyd (Hercule Poirot #4),44.10,4
6,A Time of Torment (Charlie Parker #14),48.35,5
7,Murder at the 42nd Street Library (Raymond Amb...,54.36,4
8,What Happened on Beale Street (Secrets of the ...,25.37,5
9,The Bachelor Girl's Guide to Murder (Herringfo...,52.30,5



Query 5 result:


,title,price_gbp,price_inr
0,Blood Defense (Samantha Brinkman #1),20.30,2141.65
1,"Love, Lies and Spies",20.55,2168.02
2,Between Shades of Gray,20.79,2193.34
3,Delivering the Truth (Quaker Midwife Mystery #1),20.89,2203.90
4,Voyager (Outlander #3),21.07,2222.89
5,The Silkworm (Cormoran Strike #2),23.05,2431.78
6,The Road to Little Dribbling: Adventures of an...,23.21,2448.66
7,Career of Evil (Cormoran Strike #3),24.72,2607.96
8,The Mysterious Affair at Styles (Hercule Poiro...,24.80,2616.40
9,What Happened on Beale Street (Secrets of the ...,25.37,2676.54



Books table:


...


JOIN result using pandas merge:


,title,category_name,price_gbp,rating,in_stock
0,A Flight of Arrows (The Pathfinders #2),Historical Fiction,55.53,5,1
1,The Bachelor Girl's Guide to Murder (Herringfo...,Mystery,52.30,5,1
2,A Time of Torment (Charlie Parker #14),Mystery,48.35,5,1
3,While You Were Mine,Historical Fiction,41.32,5,1
4,The Red Tent,Historical Fiction,35.66,5,1
5,Mrs. Houdini,Historical Fiction,30.25,5,1
6,The Passion of Dolssa,Historical Fiction,28.32,5,1
7,"1,000 Places to See Before You Die",Travel,26.08,5,1
8,What Happened on Beale Street (Secrets of the ...,Mystery,25.37,5,1
9,The Silkworm (Cormoran Strike #2),Mystery,23.05,5,1



Do both approaches produce the same result?
True

SQL JOIN vs pandas merge:


,SQL_title,SQL_category_name,SQL_price_gbp,SQL_rating,SQL_in_stock,Pandas_title,Pandas_category_name,Pandas_price_gbp,Pandas_rating,Pandas_in_stock
0,A Flight of Arrows (The Pathfinders #2),Historical Fiction,55.53,5,1,A Flight of Arrows (The Pathfinders #2),Historical Fiction,55.53,5,1
1,The Bachelor Girl's Guide to Murder (Herringfo...,Mystery,52.30,5,1,The Bachelor Girl's Guide to Murder (Herringfo...,Mystery,52.30,5,1
2,A Time of Torment (Charlie Parker #14),Mystery,48.35,5,1,A Time of Torment (Charlie Parker #14),Mystery,48.35,5,1
3,While You Were Mine,Historical Fiction,41.32,5,1,While You Were Mine,Historical Fiction,41.32,5,1
4,The Red Tent,Historical Fiction,35.66,5,1,The Red Tent,Historical Fiction,35.66,5,1
5,Mrs. Houdini,Historical Fiction,30.25,5,1,Mrs. Houdini,Historical Fiction,30.25,5,1
6,The Passion of Dolssa,Historical Fiction,28.32,5,1,The Passion of Dolssa,Historical Fiction,28.32,5,1
7,"1,000 Places to See Before You Die",Travel,26.08,5,1,"1,000 Places to See Before You Die",Travel,26.08,5,1
8,What Happened on Beale Street (Secrets of the ...,Mystery,25.37,5,1,What Happened on Beale Street (Secrets of the ...,Mystery,25.37,5,1
9,The Silkworm (Cormoran Strike #2),Mystery,23.05,5,1,The Silkworm (Cormoran Strike #2),Mystery,23.05,5,1



Database connection closed.


## Module 1 Completion

The data pipeline completes the following stages:
1: Scraped book data from multiple categories
2:Cleaned price, rating, and availability fields
3:Converted GBP prices to INR using the fixed rate of 1 GBP = 105.50 INR
4:Stored the cleaned data in a normalized SQLite database
5:Executed SQL queries covering the required SQL operations
6:Reproduced the SQL JOIN using pandas merge
7:Compared the SQL and pandas results for equivalence